In [ ]:
"""
A Gradio demo web application developed for Cardiac MRI segmentation.
"""

# Import libraries
import random
import gradio as gr
import torch
import numpy as np
import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from skimage import io
from io import BytesIO
from src.org_unet import UNet
from src.residual_unet import ResidualUNet
from src.attention_unet import AttentionUNetV3
from src.feature_pyramid_unet import FeaturePyramidUNet
from src.feedback_resunet import FeedbackResUNet
from src.transUnet import TransformerUNet
%matplotlib inline

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')


def load_model(checkpoint_path, model_class):
    """
    Load a model from a given checkpoint.

    Parameters:
        checkpoint_path (Path): Path to the model checkpoint.
        model_class: Class of the model architecture.

    Returns:
        torch.nn.Module: Loaded model.
    """
    model_ = model_class()
    checkpoint = torch.load(checkpoint_path)
    model_.load_state_dict(checkpoint)
    model_ = model_.eval()
    return model_


def preprocess_image(image):
    """
    Preprocess the image for segmentation.
    Parameters:
        image (np.array): Image to be preprocessed.
    Returns:
        image (np.array): Preprocessed image.
    """
    # print(f"input_image: shape {image.shape} | dtype {image.dtype}")

    image = (image * 255).astype(np.uint8)  # Scale to [0, 255]
    # print(f"input_image after re-scaling: shape {image.shape} | dtype {image.dtype}")

    # Normalize image
    normalize_transform = A.Normalize(mean=0.5, std=0.5, max_pixel_value=1.0)
    normalized_image = normalize_transform(image=image)
    image = torch.from_numpy(normalized_image['image'])
    # print(f"input_image after normalizing: shape {image.shape} | dtype {image.dtype}")

    return image.unsqueeze(-1) # Add channel dimension


def visualize_instance_seg_mask(mask):
    """
    Visualizes an instance segmentation mask in grayscale.

    Args:
        mask (numpy.ndarray): 2D array where each pixel corresponds to a class label.

    Returns:
        numpy.ndarray: Grayscale visualization of the mask.
    """
    # Normalize the mask to [0, 255] range
    mask_normalized = (mask - mask.min()) / (mask.max() - mask.min()) * 255
    mask_normalized = mask_normalized.astype(np.uint8)  # Convert to uint8

    # Create a grayscale image
    image = np.stack([mask_normalized] * 3, axis=-1)  # Convert to 3-channel grayscale
    # image = np.stack([mask] * 1, axis=-1)  # Convert to 1-channel grayscale

    return image

def visualize_instance_seg_mask_2(mask):
    label2color = {
        label: (random.randint(0, 1), random.randint(0, 255), random.randint(0, 255)) for label in range(4)
    }
    image = np.zeros((mask.shape[0], mask.shape[1], 3))
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            image[i, j, :] = label2color[mask[i, j]]
    image = image / 255
    return image

def segment_cmri(image):
    """
    Segment the image using a trained model.
    Parameters:
        image (np.array): Image to be segmented.
    Returns:
        results (np.array): Segmented image.
    """
    # Prepare visualization grid
    # fig, axes = plt.subplots(1, 2, figsize=(3 * 2, 8))
    # axes[0].imshow(image, cmap="gray")
    # axes[0].set_title("Original Image")
    # axes[0].axis("off")

    # Load model
    model_path = Path(f'./models/checkpoints/feat_pyramid_unet_img_slices_with_ratios_v4_20241224021156-rumbling-dog-720/checkpoint_epoch{23}.pth')
    model = load_model(checkpoint_path=model_path, model_class=FeaturePyramidUNet)
    # model_path = Path(f'./models/checkpoints/org_unet_img_slices_with_ratios_v4_20250101083911-handsome-moose-917/checkpoint_epoch{20}.pth')
    # model = load_model(checkpoint_path=model_path, model_class=UNet)
    # model = FeaturePyramidUNet()
    # model = UNet()
    # checkpoint = torch.load(model_path)
    # model.load_state_dict(checkpoint)
    # model = model.eval()

    # Preprocess the image
    # Convert NumPy array to PIL image
    # pil_img = Image.fromarray(image)

    # Convert PIL image to BytesIO
    # img_bytes = BytesIO()
    # pil_img.save(img_bytes, format='PNG')
    # img_bytes.seek(0)

    # Use skimage.io.imread to read from BytesIO
    # image = io.imread(img_bytes, as_gray=True)
    image = preprocess_image(image)
    # image = (image * 255).astype(np.uint8)  # Scale to [0, 255]
    # normalize_transform = A.Normalize(mean=0.5, std=0.5, max_pixel_value=1.0) # Normalize image
    # normalized_image = normalize_transform(image=image)
    # image = torch.from_numpy(normalized_image['image'])
    # image = image.unsqueeze(-1) # Add channel dimension

    with torch.no_grad():
        # Move the input image to the same device as the model
        image = image.unsqueeze(0)  # Add batch dimension
        # print(f"input_image after unsqueeze-II (add batch dim): shape {image.shape} | dtype {image.dtype}")
        image = image.permute(0, 3, 1, 2)  # Ensure proper format for PyTorch models
        # print(f"permuted input_image: shape {image.shape} | dtype {image.dtype}")
        image = image.to(device, dtype=torch.float32, memory_format=torch.channels_last)
        # print(f"device transformed input_image: shape {image.shape} | dtype {image.dtype}")

        model = model.to(device)

        prediction = model(image)  # Get model prediction
        prediction = torch.argmax(prediction, dim=1).squeeze(0).cpu().numpy()
        # print(f"prediction: shape {prediction.shape} | dtype {prediction.dtype}")

        # Ensure the prediction array is in the correct format
        # if prediction.dtype != np.uint8:
        #     prediction = (prediction * 255).astype(np.uint8)  # Normalize if needed
        # Convert to a PIL image
        # pil_image = Image.fromarray(prediction)

        # results = visualize_instance_seg_mask(prediction)
        results = visualize_instance_seg_mask_2(prediction)

        # Convert to RGB colormap for better visualization
        # cmap = cm.get_cmap("tab10", 4)  # 4-class visualization with distinct colors
        # prediction_colored = cmap(prediction / prediction.max())  # Normalize & apply colormap
        # results = (prediction_colored[:, :, :3] * 255).astype(np.uint8)  # Convert to 3-channel

        # Normalize for visualization
        # prediction_normalized = (prediction / prediction.max() * 255).astype(np.uint8)

        # Convert to grayscale PIL image
        # prediction_image = Image.fromarray(prediction_normalized, mode='L')

    # axes[1].imshow(prediction, cmap="gray", interpolation="none")
    # axes[1].set_title("Segmented Image")
    # axes[1].axis("off")
    #
    # plt.tight_layout()
    # plt.show()

    return results

# Gradio Demo
demo = gr.Interface(
    fn=segment_cmri,
    # inputs=[gr.Image(image_mode='L', label="Input Image")],
    inputs=[gr.Image(type="numpy", image_mode='L', label="Input Image")],
    # inputs=gr.Image(type="pil", image_mode='L', label="Input Image"),
    # outputs=[gr.Image(type="numpy", image_mode='L', label="Segmented Image")],
    outputs="image",
    # outputs=gr.Image(type="pil", label="Segmented Image"),
    title="CMRI Segmentation Demo",
    examples=[["../data/ACDC/img_slices_with_ratios_v4/testing/labeled_-1/images/subject136_frame01_slice01_ras.png"]]
)

demo.launch(share=False, debug=True)

In [ ]:
from skimage import io

img_path = Path(f'../data/ACDC/img_slices_with_ratios_v4/testing/labeled_-1/images/subject136_frame01_slice01_ras.png')
img = io.imread(img_path, as_gray=True)

prediction_image = segment_cmri(img)